# Face Recognition — Base d'embeddings InsightFace (LFW Top-20)

**Projet** : Système de Reconnaissance Faciale — PFE Sonatel Academy  
**Auteur** : Ibrahima Gabar Diop  

## Pipeline
```
InsightFace buffalo_l
  ├── Détection   : RetinaFace  (det_10g.onnx)
  └── Embeddings  : ArcFace ResNet50 (w600k_r50.onnx) — 512 dimensions
```

Ce notebook :
1. Sélectionne les 20 personnes LFW les plus représentées
2. Génère un embedding ArcFace moyen par personne
3. Exporte `face_embeddings.npz` + une image de référence par personne
4. Ces fichiers sont directement utilisables par `app/recognition.py`


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'insightface', 'onnxruntime-gpu', '-q'], check=True)


In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from insightface.app import FaceAnalysis

LFW_DIR      = Path('/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled')
CSV_ALLNAMES = Path('/kaggle/input/lfw-dataset/lfw_allnames.csv')
OUTPUT_DIR   = Path('/kaggle/working')
REFS_DIR     = OUTPUT_DIR / 'face_refs'   # images de référence

N_CLASSES  = 20
MIN_IMAGES = 30
N_REFS_PER_PERSON = 5   # embeddings moyennés sur N images
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REFS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Sélection des classes


In [ ]:
df_all = pd.read_csv(CSV_ALLNAMES)
top20  = df_all[df_all['images'] >= MIN_IMAGES].sort_values('images', ascending=False).head(N_CLASSES)
print(top20[['name','images']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top20['name'][::-1], top20['images'][::-1], color='steelblue')
ax.set_xlabel('Nombre d images')
ax.set_title('LFW Top-20 — distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_balance.png', dpi=150)
plt.show()


## 2. Chargement InsightFace


In [ ]:
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace buffalo_l chargé')


## 3. Génération des embeddings


In [ ]:
embeddings_db = {}
failed = []

for _, row in top20.iterrows():
    person = row['name']
    imgs   = sorted((LFW_DIR / person).glob('*.jpg'))
    random.shuffle(imgs)
    sample = imgs[:N_REFS_PER_PERSON]

    embs = []
    best_img_path = None
    best_area     = 0

    for img_path in sample:
        img   = cv2.imread(str(img_path))
        faces = face_app.get(img)
        if not faces:
            continue
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        embs.append(face.embedding)
        area = (face.bbox[2]-face.bbox[0]) * (face.bbox[3]-face.bbox[1])
        if area > best_area:
            best_area     = area
            best_img_path = img_path

    if not embs:
        failed.append(person)
        print(f'  FAIL  {person}')
        continue

    mean_emb = np.mean(embs, axis=0)
    mean_emb = mean_emb / np.linalg.norm(mean_emb)
    embeddings_db[person] = mean_emb

    # Copier la meilleure image de référence
    shutil.copy(best_img_path, REFS_DIR / f'{person}.jpg')
    print(f'  OK    {person} — {len(embs)} embeddings moyennés')

print(f'\nBase construite : {len(embeddings_db)} identités, {len(failed)} échecs')


## 4. Export


In [ ]:
# Sauvegarder les embeddings
npz_path = OUTPUT_DIR / 'face_embeddings.npz'
np.savez(str(npz_path), **embeddings_db)
print(f'Embeddings sauvegardés : {npz_path}')

# Vérification
loaded = np.load(str(npz_path))
print(f'Identités : {list(loaded.keys())}')
print(f'Dimension embedding : {loaded[list(loaded.keys())[0]].shape}')


In [ ]:
# Visualisation des images de référence exportées
ref_imgs = sorted(REFS_DIR.glob('*.jpg'))
cols = 5
rows = (len(ref_imgs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
axes = axes.flatten()
for i, img_path in enumerate(ref_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(img_path.stem[:18], fontsize=8)
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Images de référence — LFW Top-20', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reference_faces.png', dpi=150)
plt.show()


## 5. Instructions d'utilisation

Télécharger depuis `/kaggle/working` :
- `face_embeddings.npz` → copier dans `data/`
- `face_refs/*.jpg`     → copier dans `data/faces/`

L'application chargera automatiquement `data/faces/embeddings.pkl` (cache généré au 1er démarrage).
Les 20 personnes LFW seront immédiatement reconnues.


In [ ]:
print('=' * 55)
print('  RÉSUMÉ')
print('=' * 55)
print(f'  Modèle       : InsightFace buffalo_l')
print(f'  Détecteur    : RetinaFace (det_10g.onnx)')
print(f'  Embeddings   : ArcFace ResNet50 (w600k_r50.onnx)')
print(f'  Identités    : {len(embeddings_db)}')
print(f'  Dim embedding: 512')
print(f'  Fichiers     :')
print(f'    face_embeddings.npz  -> data/')
print(f'    face_refs/*.jpg      -> data/faces/')
print('=' * 55)
